In [1]:
import os
import shutil
import json
import torch
import time
import nltk
import numpy as np
from google.colab import drive

# 1. CÀI ĐẶT THƯ VIỆN & CẤP QUYỀN NUMPY
os.system("pip install -q peft")
from peft import PeftModel
from transformers import BartTokenizer, BartForConditionalGeneration
from datasets import Dataset

try:
    reconstruct_func = np._core.multiarray._reconstruct if hasattr(np, '_core') else np.core.multiarray._reconstruct
    torch.serialization.add_safe_globals([
        reconstruct_func,
        np.ndarray,
        np.dtype,
    ])
except AttributeError:
    pass

if not hasattr(torch, 'original_load_func'):
    torch.original_load_func = torch.load

def safe_load_override(*args, **kwargs):
    if 'weights_only' in kwargs:
        del kwargs['weights_only']
    return torch.original_load_func(*args, weights_only=False, **kwargs)

torch.load = safe_load_override

# 2. KẾT NỐI DRIVE & TẢI DỮ LIỆU
os.environ['KAGGLE_USERNAME'] = "phankhaclap"
os.environ['KAGGLE_KEY'] = "0ba946628cb1f5acb76ecd357f590e95"

drive.mount('/content/drive')
FINAL_SAVE_PATH = "/content/drive/My Drive/BART_Base_Spider_CRP_LoRA"

print(">>> [1/4] Kiểm tra và tải dữ liệu...")
if not os.path.exists('spider_data'):
    !pip install -q kaggle
    !kaggle datasets download -d jeromeblanchet/yale-universitys-spider-10-nlp-dataset
    !unzip -q yale-universitys-spider-10-nlp-dataset.zip -d temp_spider
    if os.path.exists("temp_spider/spider"):
        shutil.move("temp_spider/spider", "spider_data")
        shutil.rmtree('temp_spider')
        !wget -q https://raw.githubusercontent.com/taoyds/spider/master/evaluation.py
        !wget -q https://raw.githubusercontent.com/taoyds/spider/master/process_sql.py
        nltk.download('punkt', quiet=True)
        nltk.download('punkt_tab', quiet=True)

# Tải thêm tập Spider-Realistic
if not os.path.exists('spider_data/spider-realistic.json'):
    !wget -q -O spider_data/spider-realistic.json "https://zenodo.org/records/5205322/files/spider-realistic.json?download=1"

# 3. TIỀN XỬ LÝ SCHEMA
with open('spider_data/tables.json', 'r') as f:
    table_data = json.load(f)
schema_map = {db['db_id']: db for db in table_data}

def get_crp_schema(db_id):
    if db_id not in schema_map: return ""
    db = schema_map[db_id]
    ddl_statements = []
    for i, table_name in enumerate(db['table_names_original']):
        col_defs = [f"{c[1]} {db['column_types'][j]}" for j, c in enumerate(db['column_names_original']) if c[0] == i]
        pk_idx = db['primary_keys']
        pks = [db['column_names_original'][j][1] for j in pk_idx if db['column_names_original'][j][0] == i]
        if pks: col_defs.append(f"primary key ({', '.join(pks)})")
        for fk in db['foreign_keys']:
            if db['column_names_original'][fk[0]][0] == i:
                from_col = db['column_names_original'][fk[1]][1]
                to_table = db['table_names_original'][db['column_names_original'][fk[1]][0]]
                to_col = db['column_names_original'][fk[1]][1]
                col_defs.append(f"foreign key ({from_col}) references {to_table}({to_col})")
        ddl_statements.append(f"CREATE TABLE {table_name} ({', '.join(col_defs)});")
    return " ".join(ddl_statements)

def load_spider_unified(file_path):
    with open(file_path, 'r', encoding='utf-8') as f:
        data = json.load(f)
    return Dataset.from_dict({
        "question": [item["question"] for item in data],
        "query": [item["query"] for item in data],
        "db_id": [item["db_id"] for item in data]
    })

print(">>> [2/4] Đang load tập Spider-Realistic...")
val_ds = load_spider_unified("spider_data/spider-realistic.json")

# 4. LOAD MÔ HÌNH LORA (QUAN TRỌNG)
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"\n>>> [3/4] Đang tải mô hình LoRA lên {device}...")

MODEL_NAME = "facebook/bart-base"
tokenizer = BartTokenizer.from_pretrained(FINAL_SAVE_PATH)

# Load model gốc trước
base_model = BartForConditionalGeneration.from_pretrained(MODEL_NAME)
# Ghép trọng số LoRA vào model gốc
model = PeftModel.from_pretrained(base_model, FINAL_SAVE_PATH)
model = model.to(device)
model.eval()

# 5. CHẠY INFERENCE (BATCH) & ĐÁNH GIÁ
print("\n>>> [4/4] Bắt đầu Inference (Batch Processing)...")
predictions, gold_lines = [], []
input_texts = []

for item in val_ds:
    input_texts.append(f"translate to SQL: {item['question']} | Schema: {get_crp_schema(item['db_id'])}")
    gold_lines.append(f"{item['query']}\t{item['db_id']}\n")

batch_size = 16
total_batches = len(input_texts)
print(f"Bắt đầu dịch {total_batches} câu hỏi sang SQL...")
start_time = time.time()

for i in range(0, total_batches, batch_size):
    batch_texts = input_texts[i : i + batch_size]
    inputs = tokenizer(batch_texts, return_tensors="pt", padding=True, truncation=True, max_length=1024)

    input_ids = inputs.input_ids.to(model.device)
    attention_mask = inputs.attention_mask.to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            input_ids=input_ids,
            attention_mask=attention_mask,
            max_length=128,
            num_beams=4,
            early_stopping=True
        )

    batch_preds = tokenizer.batch_decode(outputs, skip_special_tokens=True)
    predictions.extend([pred + "\n" for pred in batch_preds])

    print(f"\rĐã xử lý: {min(i + batch_size, total_batches)}/{total_batches}", end="")

print(f"\n✅ Xong Inference trong {time.time() - start_time:.2f} giây!")

with open('pred.txt', 'w') as f: f.writelines(predictions)
with open('gold.txt', 'w') as f: f.writelines(gold_lines)

print("\n>>> KẾT QUẢ ĐÁNH GIÁ:")
!sed -i 's/conn = sqlite3.connect(db)/conn = sqlite3.connect(db)\n    conn.text_factory = lambda b: b.decode(errors="ignore")/' evaluation.py
!python evaluation.py --gold gold.txt --pred pred.txt --db spider_data/database --table spider_data/tables.json --etype all

Mounted at /content/drive
>>> [1/4] Kiểm tra và tải dữ liệu...
Dataset URL: https://www.kaggle.com/datasets/jeromeblanchet/yale-universitys-spider-10-nlp-dataset
License(s): unknown
100% 96.0M/96.0M [00:00<00:00, 169MB/s]

>>> [2/4] Đang load tập Spider-Realistic...

>>> [3/4] Đang tải mô hình LoRA lên cuda...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/558M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/259 [00:00<?, ?it/s]


>>> [4/4] Bắt đầu Inference (Batch Processing)...
Bắt đầu dịch 508 câu hỏi sang SQL...
Đã xử lý: 508/508
✅ Xong Inference trong 96.77 giây!

>>> KẾT QUẢ ĐÁNH GIÁ:
medium pred: SELECT Name ,  Country ,  Age FROM singer ORDER BY Age ASC
medium gold: SELECT name ,  country ,  age FROM singer ORDER BY age DESC

medium pred: SELECT count(*) FROM concert WHERE YEAR  =  2014 OR YEAR != 2015
medium gold: SELECT count(*) FROM concert WHERE YEAR  =  2014 OR YEAR  =  2015

medium pred: SELECT count(*) FROM concert WHERE YEAR  =  2014 OR YEAR != 2015
medium gold: SELECT count(*) FROM concert WHERE YEAR  =  2014 OR YEAR  =  2015

eval_err_num:1
extra pred: SELECT stadium_name ,  capacity FROM concert WHERE YEAR  >=  2014 GROUP BY stadium_id ORDER BY count(*) DESC LIMIT 1
extra gold: SELECT T2.name ,  T2.capacity FROM concert AS T1 JOIN stadium AS T2 ON T1.stadium_id  =  T2.stadium_id WHERE T1.year  >=  2014 GROUP BY T2.stadium_id ORDER BY count(*) DESC LIMIT 1

eval_err_num:2
extra pred: SELECT T1